# [17.1] Checkpoint Archaeology and Mechanism Emergence - Exercises

Implement GT-0 checkpoint-archaeology helpers, then use visible tests to check first crossing, stable emergence, phase jumps, controls, toy developmental comparisons, and a live train/save/reload checkpoint smoke path. The real CUDA checkpoint preflight is validated through the committed report and the solution notebook.

## Setup

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import sys
import tempfile
from pathlib import Path

import torch as t

chapter = "chapter17_training_dynamics"
section = "part1_checkpoint_archaeology"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_checkpoint_archaeology.tests as tests
import part1_checkpoint_archaeology.utils as utils

from arena_ext.training_dynamics import (
    DevelopmentalComparisonReport,
    MechanismEmergenceReport,
    PhaseTransitionReport,
    RandomControlReport,
    toy_training_trajectories,
)

MAIN = __name__ == "__main__"

## First and Stable Crossings

Return checkpoint steps, not tensor indices. Stable emergence requires consecutive checkpoints above threshold.

In [ ]:
def _validate_checkpoint_series(
    steps: t.Tensor,
    values: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    steps = t.as_tensor(steps).flatten().long()
    values = t.as_tensor(values).flatten().float()
    raise NotImplementedError()


def first_threshold_crossing(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
) -> int | None:
    raise NotImplementedError()


def stable_threshold_step(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
    min_consecutive: int = 2,
) -> int | None:
    raise NotImplementedError()


tests.test_first_threshold_crossing_finds_first_crossing_and_validates_inputs(
    first_threshold_crossing
)
tests.test_stable_threshold_step_requires_consecutive_checkpoints(
    stable_threshold_step
)

## Emergence Reports

A useful report carries the claim evidence: first crossing, stable crossing, peak, monotonicity, and final pass/fail status.

In [ ]:
def monotonicity_violations(values: t.Tensor, *, tolerance: float = 0.0) -> int:
    raise NotImplementedError()


def mechanism_emergence_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    threshold: float = 0.6,
    min_consecutive: int = 2,
    max_monotonicity_violations: int = 1,
) -> MechanismEmergenceReport:
    raise NotImplementedError()


tests.test_mechanism_emergence_report_tracks_peak_and_monotonicity(
    mechanism_emergence_report
)

## Phase Transitions and Controls

Adjacent jumps are a warning signal. Random or label-shuffled controls are a falsification gate.

In [ ]:
def phase_transition_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    min_jump: float = 0.25,
) -> PhaseTransitionReport:
    raise NotImplementedError()


def random_control_report(
    values: t.Tensor,
    *,
    metric_name: str = "random_control",
    max_allowed_value: float = 0.3,
) -> RandomControlReport:
    raise NotImplementedError()


tests.test_phase_transition_report_detects_largest_adjacent_jump(
    phase_transition_report
)
tests.test_random_control_report_rejects_overstrong_control(
    random_control_report
)

## Developmental Comparison

Keep the random control visible, but do not rank it as a model family.

In [ ]:
def developmental_comparison_report(
    steps: t.Tensor,
    family_values: dict[str, t.Tensor],
    *,
    threshold: float = 0.6,
    min_consecutive: int = 2,
    control_name: str = "random_control",
) -> DevelopmentalComparisonReport:
    raise NotImplementedError()


tests.test_developmental_comparison_excludes_random_control_from_ordering(
    developmental_comparison_report
)

## Live Checkpoint Archaeology

Train a tiny mod-13 addition model, save scheduled checkpoints, reload each checkpoint before measuring it, and run a random-label checkpoint control. This is exhaustive finite-domain evidence over all 169 mod-13 input pairs, not OOD generalization.

In [ ]:
def _train_save_reload_modular_run(
    checkpoint_dir: Path,
    *,
    device: t.device,
    seed: int,
    random_labels: bool = False,
) -> dict:
    t.manual_seed(seed)
    input_pairs, true_labels = utils.modular_addition_table(device=device)
    generator = t.Generator(device=device).manual_seed(seed + 1000)
    train_labels = true_labels
    if random_labels:
        train_labels = t.randint(
            0,
            utils.LIVE_MODULAR_ARCHAEOLOGY_MODULUS,
            true_labels.shape,
            generator=generator,
            device=device,
        )

    model = utils.TinyModularAdditionMLP().to(device)
    optimizer = t.optim.AdamW(
        model.parameters(),
        lr=utils.LIVE_MODULAR_ARCHAEOLOGY_LR,
        weight_decay=1e-3,
    )

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_paths: list[Path] = []
    checkpoint_steps = set(utils.LIVE_MODULAR_ARCHAEOLOGY_CHECKPOINT_STEPS)
    for step in range(utils.LIVE_MODULAR_ARCHAEOLOGY_STEPS + 1):
        if step in checkpoint_steps:
            raise NotImplementedError("save this checkpoint and record its path")
        if step == utils.LIVE_MODULAR_ARCHAEOLOGY_STEPS:
            break
        optimizer.zero_grad(set_to_none=True)
        loss = t.nn.functional.cross_entropy(model(input_pairs), train_labels)
        loss.backward()
        optimizer.step()

    reloaded_accuracies: list[float] = []
    reloaded_losses: list[float] = []
    for path in checkpoint_paths:
        raise NotImplementedError("reload this checkpoint and append accuracy/loss")

    return {
        "steps": t.tensor(utils.LIVE_MODULAR_ARCHAEOLOGY_CHECKPOINT_STEPS, device=device),
        "accuracies": t.tensor(reloaded_accuracies, device=device),
        "losses": t.tensor(reloaded_losses, device=device),
        "checkpoint_count": len(checkpoint_paths),
        "checkpoint_total_bytes": sum(path.stat().st_size for path in checkpoint_paths),
    }


def live_checkpoint_archaeology_smoke_test(
    checkpoint_root: Path | None = None,
    *,
    device: str | t.device = "cpu",
    seed: int = 0,
) -> dict:
    raise NotImplementedError()


tests.test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls(
    live_checkpoint_archaeology_smoke_test
)

## Notebook Contract

Expose JSON-serializable smoke-test outputs for the report runner, including the live CPU checkpoint smoke path.

In [ ]:
def checkpoint_emergence_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return mechanism_emergence_report(
        steps,
        trajectories["autoregressive"],
        metric_name="induction_probe_accuracy",
        threshold=0.6,
        min_consecutive=2,
    ).__dict__.copy()


def phase_transition_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return phase_transition_report(
        steps,
        trajectories["autoregressive"],
        metric_name="induction_probe_accuracy",
        min_jump=0.3,
    ).__dict__.copy()


def random_control_smoke_test() -> dict:
    _, trajectories = toy_training_trajectories()
    return random_control_report(
        trajectories["random_control"],
        metric_name="label_shuffled_probe",
        max_allowed_value=0.2,
    ).__dict__.copy()


def developmental_comparison_smoke_test() -> dict:
    steps, trajectories = toy_training_trajectories()
    return developmental_comparison_report(
        steps,
        trajectories,
        threshold=0.6,
        min_consecutive=2,
    ).__dict__.copy()


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "checkpoint_emergence": checkpoint_emergence_smoke_test(),
        "phase_transition": phase_transition_smoke_test(),
        "random_control": random_control_smoke_test(),
        "developmental_comparison": developmental_comparison_smoke_test(),
        "live_checkpoint_archaeology": live_checkpoint_archaeology_smoke_test(device="cpu"),
    }


tests.test_checkpoint_emergence_smoke_test(checkpoint_emergence_smoke_test)
tests.test_phase_transition_smoke_test(phase_transition_smoke_test)
tests.test_random_control_smoke_test(random_control_smoke_test)
tests.test_developmental_comparison_smoke_test(developmental_comparison_smoke_test)
tests.test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls(
    live_checkpoint_archaeology_smoke_test
)
tests.test_notebook_contract(run_smoke_test)

## CUDA Report Check

The real GPU path trains a tiny modular-addition MLP, writes/reloads checkpoints, and rejects a random-label control. It reports exhaustive finite-domain coverage over all 169 mod-13 pairs, not OOD generalization. It is checked in the solution notebook and by pytest through the committed report.

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
